# R1-04 · Limpieza de datos — Profundización (opcional) 🔬

**Formación Pública — Rama R1 · Analista de Datos · Notebook de profundización**

Opcional y de **transferencia**: en la lección los errores eran inyectados; aquí los faltantes
son **reales**. PM2.5 diario de 14 estaciones de monitoreo SINCA, enero–septiembre de 2026
(`aire_nuevos.csv`). Vas a diagnosticar por estación, ver si faltan **al azar o en bloque**,
medir el sesgo de rellenar con cero, hacer una **comparación justa** y escribir la
**nota metodológica** con tu decisión.

> Cada ejercicio termina en una **celda de chequeo** con ✅ / pista.

In [ ]:
import pandas as pd, os, urllib.request
CSV = "aire_nuevos.csv"
if not os.path.exists(CSV):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/aire_nuevos.csv", CSV)
aire = pd.read_csv(CSV, parse_dates=["fecha"])
print("filas:", aire.shape, "| estaciones:", aire["estacion"].nunique(),
      "| desde", aire["fecha"].min().date(), "hasta", aire["fecha"].max().date())
print(aire.isna().sum())

## 1. ¿Qué estaciones tienen más días sin dato?

Datos nuevos y **reales**: PM2.5 diario de 14 estaciones SINCA entre enero y septiembre de 2026. Aquí nadie inyectó errores: los huecos son de verdad (equipos en mantención, fallas de transmisión, datos no validados). Primero, el diagnóstico: **% de días sin PM2.5 por estación**.

### ✍️ Ejercicio 1 — Porcentaje de faltantes por estación

In [ ]:
pct_faltante = (aire.assign(falta=aire["pm25"].isna())
                    .groupby("estacion")["falta"].mean() * 100).sort_values(ascending=False)
peor = pct_faltante.idxmax()
print(pct_faltante.round(1))
print("peor estación:", peor)

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    _esp = aire["pm25"].isna().groupby(aire["estacion"]).mean() * 100
    assert abs(pct_faltante.max() - _esp.max()) < 1e-9, "Revisa el cálculo del porcentaje"
    assert list(pct_faltante.values) == sorted(pct_faltante.values, reverse=True), "Ordena de mayor a menor"
    assert peor == _esp.idxmax()
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'La media de un booleano es la proporción de True; multiplica por 100 y ordena con sort_values(ascending=False).')
    print("   Detalle:", e)

## 2. ¿Cuándo faltan? Al azar o en bloque

No es lo mismo que falten días sueltos (al azar) que **meses enteros** seguidos. Mira la peor estación mes a mes: ¿cuántos días hay y cuántos faltan?

### ✍️ Ejercicio 2 — Faltantes por mes en la peor estación

In [ ]:
est = aire[aire["estacion"] == peor].copy()
est["mes"] = est["fecha"].dt.month
por_mes = est.assign(falta=est["pm25"].isna()).groupby("mes")["falta"].agg(["size", "sum"])
meses_vacios = list(por_mes.index[por_mes["sum"] == por_mes["size"]])
primer_dato = est.loc[est["pm25"].notna(), "fecha"].min()
print(por_mes)
print("meses completamente vacíos:", meses_vacios, "| primer dato:", primer_dato.date())

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    _e = aire[(aire["estacion"] == peor)]
    _m = _e.assign(falta=_e["pm25"].isna()).groupby(_e["fecha"].dt.month)["falta"].agg(["size", "sum"])
    assert list(meses_vacios) == list(_m.index[_m["sum"] == _m["size"]]), "Filtra por_mes.index donde sum == size"
    assert pd.Timestamp(primer_dato) == _e.loc[_e["pm25"].notna(), "fecha"].min(), "Busca el mínimo de fecha donde pm25 no es nulo"
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'list(por_mes.index[por_mes["sum"] == por_mes["size"]]) y est.loc[est["pm25"].notna(), "fecha"].min().')
    print("   Detalle:", e)

## 3. Nunca rellenes con cero

Un cero en PM2.5 significa «aire perfectamente limpio», no «no hay dato». Compara el promedio honesto (ignorando los `NaN`, que es lo que hace `.mean()`) con el de alguien que rellenó con 0.

### ✍️ Ejercicio 3 — Mide el sesgo de rellenar con 0

In [ ]:
prom_honesto = est["pm25"].mean()
prom_con_ceros = est["pm25"].fillna(0).mean()
sesgo_pct = 100 * (prom_con_ceros - prom_honesto) / prom_honesto
print(f"honesto: {prom_honesto:.1f} µg/m³ | con ceros: {prom_con_ceros:.1f} µg/m³ | sesgo: {sesgo_pct:.0f}%")

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    assert abs(prom_honesto - est["pm25"].mean()) < 1e-9
    assert abs(prom_con_ceros - est["pm25"].fillna(0).mean()) < 1e-9
    assert abs(sesgo_pct - 100 * (prom_con_ceros - prom_honesto) / prom_honesto) < 1e-9
    assert sesgo_pct < -20, "Rellenar con 0 debería bajar mucho el promedio"
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'est["pm25"].mean() y est["pm25"].fillna(0).mean().')
    print("   Detalle:", e)

## 4. La comparación justa

Pero incluso el promedio «honesto» engaña si los huecos caen en una **época del año**. En Santiago el PM2.5 sube en otoño-invierno. Si la peor estación solo tiene datos desde mayo, su promedio es **de invierno**. Compárala con Cerro Navia (también en Santiago): primero ingenuamente, después **solo en los días en que ambas midieron**.

### ✍️ Ejercicio 4 — Compara en los mismos días

In [ ]:
otra = aire[aire["estacion"] == "Cerro Navia"]
ingenua = {peor: est["pm25"].mean(), "Cerro Navia": otra["pm25"].mean()}
ambas = pd.merge(est[["fecha", "pm25"]], otra[["fecha", "pm25"]], on="fecha",
                 suffixes=("_peor", "_cn")).dropna()
justa = {peor: ambas["pm25_peor"].mean(), "Cerro Navia": ambas["pm25_cn"].mean()}
print("ingenua:", {k: round(float(v), 1) for k, v in ingenua.items()})
print("justa (", len(ambas), "días comunes):", {k: round(float(v), 1) for k, v in justa.items()})
mas_alta_ingenua = max(ingenua, key=ingenua.get)
mas_alta_justa = max(justa, key=justa.get)
print("más contaminada — ingenua:", mas_alta_ingenua, "| justa:", mas_alta_justa)

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    assert abs(justa[peor] - ambas["pm25_peor"].mean()) < 1e-9
    assert abs(justa["Cerro Navia"] - ambas["pm25_cn"].mean()) < 1e-9
    assert mas_alta_justa == max(justa, key=justa.get)
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'justa = {peor: ambas["pm25_peor"].mean(), "Cerro Navia": ambas["pm25_cn"].mean()}.')
    print("   Detalle:", e)

## 5. Faltante estructural: «no se mide» no es «se perdió»

La temperatura falta en muchas filas, pero no por fallas: varias estaciones **no tienen sensor meteorológico**. Eso es un faltante **estructural** y se documenta distinto: no hay nada que recuperar ni que corregir.

### ✍️ Ejercicio 5 — Encuentra las estaciones sin meteorología

In [ ]:
sin_meteo = sorted(aire.groupby("estacion")["temp_media"].apply(lambda s: s.isna().all())
                   .loc[lambda s: s].index)
print(len(sin_meteo), "estaciones sin temperatura en todo el período:", sin_meteo)

In [ ]:
# ── Celda de chequeo · Ejercicio 5 ──────────────────────────────────────────
try:
    _t = aire.groupby("estacion")["temp_media"].count()
    assert sorted(sin_meteo) == sorted(_t[_t == 0].index), "Son las estaciones con 0 valores de temp_media"
    print("✅ Ejercicio 5: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'Agrupa por estación, revisa si isna().all() y quédate con las que dan True.')
    print("   Detalle:", e)

## 6. Documentar la decisión (la nota metodológica)

Ahora decide, con una regla **escrita y pareja para todas**: menos de 10% de faltantes → *usar*; entre 10% y 30% → *usar con advertencia*; más de 30% → *excluir de comparaciones*. Una tabla así va como anexo de cualquier informe.

### ✍️ Ejercicio 6 — Arma la tabla de decisiones

In [ ]:
def decidir(pct):
    if pct < 10:
        return "usar"
    elif pct <= 30:
        return "usar con advertencia"
    return "excluir de comparaciones"

nota = pd.DataFrame({
    "dias": aire.groupby("estacion").size(),
    "dias_validos": aire.groupby("estacion")["pm25"].count(),
    "pct_faltante": (100 * aire["pm25"].isna().groupby(aire["estacion"]).mean()).round(1),
})
nota["decision"] = nota["pct_faltante"].apply(decidir)
nota = nota.sort_values("pct_faltante", ascending=False)
print(nota)

In [ ]:
# ── Celda de chequeo · Ejercicio 6 ──────────────────────────────────────────
try:
    assert decidir(5) == "usar" and decidir(20) == "usar con advertencia" and decidir(45) == "excluir de comparaciones"
    assert set(nota["decision"]) <= {"usar", "usar con advertencia", "excluir de comparaciones"}
    assert nota.loc[peor, "decision"] == "excluir de comparaciones"
    print("✅ Ejercicio 6: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "if pct < 10: ... elif pct <= 30: ... y por defecto 'excluir de comparaciones'.")
    print("   Detalle:", e)

## ✍️ Para tu informe

Escribe en tres líneas la nota metodológica que acompañaría un ranking de estaciones 2026: qué
estaciones excluiste, por qué, y qué le pasaba al ranking si no lo hacías.

> Lo honesto no es tener datos completos (casi nunca los tendrás): es **decir qué falta y qué
> hiciste con eso**.

---

*Fin de la profundización de R1-04.*